In [3]:
import io
import zipfile
import requests
import frontmatter


def _download_repo_zip(repo_owner, repo_name):
    """Download repository zip using GitHub default branch automatically."""
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball"
    headers = {"User-Agent": "aihero-course-notebook"}
    resp = requests.get(url, headers=headers, timeout=60)

    if resp.status_code != 200:
        raise Exception(
            f"Failed to download repository zip for {repo_owner}/{repo_name}: "
            f"HTTP {resp.status_code}"
        )

    return resp


def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    resp = _download_repo_zip(repo_owner, repo_name)

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
            continue

        with zf.open(file_info) as f_in:
            content = f_in.read().decode("utf-8", errors="ignore")

        # Some repos contain invalid YAML frontmatter.
        # Fallback to raw content so ingestion does not fail.
        try:
            post = frontmatter.loads(content)
            data = post.to_dict()
        except Exception as e:
            data = {
                "content": content,
                "frontmatter_error": str(e),
            }

        data["filename"] = filename
        repository_data.append(data)

    zf.close()
    return repository_data

In [4]:
dtc_langchain = read_repo_data('langchain-ai', 'langchain')
#dtc_microsoft = read_repo_data('microsoft', 'semantic-kernel')
#dtc_openai = read_repo_data('openai', 'openai-cookbook')
print(f"Langchain documents: {len(dtc_langchain)}")
#print(f"microsoft documents: {len(dtc_microsoft)}")
#print(f"openai documents: {len(dtc_openai)}")

Langchain documents: 28


In [5]:
dtc_microsoft = read_repo_data('microsoft', 'semantic-kernel')
print(f"microsoft documents: {len(dtc_microsoft)}")


microsoft documents: 231


In [ ]:
#dtc_openai = read_repo_data('openai', 'openai-cookbook') 
#print(f"openai documents: {len(dtc_openai)}")

In [ ]:
langchain-ai/langchain
microsoft/semantic-kernel
openai/openai-cookbook

In [6]:
import io
import zipfile
import requests

def count_markdown_files(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/zipball"
    resp = requests.get(url, headers={"User-Agent": "aihero-course-notebook"}, timeout=60)
    resp.raise_for_status()

    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    files = [
        f.filename for f in zf.infolist()
        if f.filename.lower().endswith((".md", ".mdx"))
    ]
    zf.close()
    return len(files), files[:20]  # sample first 20

for owner, repo in [
    ("langchain-ai", "langchain"),
    ("microsoft", "semantic-kernel"),
    #("openai", "openai-cookbook"),
]:
    n, sample = count_markdown_files(owner, repo)
    print(f"{owner}/{repo}: {n}")
    print(" sample:", sample[:5], "\n")

langchain-ai/langchain: 28
 sample: ['langchain-ai-langchain-e207685/.devcontainer/README.md', 'langchain-ai-langchain-e207685/.github/PULL_REQUEST_TEMPLATE.md', 'langchain-ai-langchain-e207685/AGENTS.md', 'langchain-ai-langchain-e207685/CLAUDE.md', 'langchain-ai-langchain-e207685/README.md'] 

microsoft/semantic-kernel: 231
 sample: ['microsoft-semantic-kernel-5f282a9/.github/ISSUE_TEMPLATE/bug_report.md', 'microsoft-semantic-kernel-5f282a9/.github/ISSUE_TEMPLATE/feature_graduation.md', 'microsoft-semantic-kernel-5f282a9/.github/ISSUE_TEMPLATE/feature_request.md', 'microsoft-semantic-kernel-5f282a9/.github/pull_request_template.md', 'microsoft-semantic-kernel-5f282a9/.github/upgrades/prompts/SemanticKernelToAgentFramework.md'] 



In [7]:
def normalize_docs(rows, repo_name):
    docs = []
    for i, row in enumerate(rows):
        text = (row.get("content") or "").strip()
        if not text:
            continue

        filename = row.get("filename", "")
        docs.append({
            "id": f"{repo_name}-{i}",
            "repo": repo_name,
            "filename": filename,
            "text": text,
        })
    return docs

In [8]:
import re


def sliding_window_text(text, size=1000, step=800):
    """Return overlapping text windows from a long string."""
    windows = []
    n = len(text)

    for start in range(0, n, step):
        end = min(start + size, n)
        chunk = text[start:end].strip()
        if chunk:
            windows.append({"start": start, "end": end, "text": chunk})
        if end == n:
            break

    return windows


def simple_fixed_chunking(text, chunk_size=1000, overlap=200):
    """Simple character-based chunking with overlap."""
    step = max(chunk_size - overlap, 1)
    windows = sliding_window_text(text, size=chunk_size, step=step)
    return [w["text"] for w in windows]


def paragraph_sliding_chunking(text, paragraph_window=3, paragraph_step=2):
    """Chunk by paragraphs, then use sliding windows across paragraphs."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if not paragraphs:
        return []

    chunks = []
    n = len(paragraphs)
    for i in range(0, n, paragraph_step):
        window = paragraphs[i:i + paragraph_window]
        if not window:
            continue
        chunk = "\n\n".join(window).strip()
        if chunk:
            chunks.append(chunk)
        if i + paragraph_window >= n:
            break

    return chunks


def section_chunking(text):
    """Chunk by markdown headings (#, ##, ###...)."""
    heading_pattern = re.compile(r"^#{1,6}\s+.*$", flags=re.MULTILINE)
    matches = list(heading_pattern.finditer(text))

    if not matches:
        # Fallback when no headings exist.
        return simple_fixed_chunking(text, chunk_size=1200, overlap=200)

    chunks = []
    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()
        if section_text:
            chunks.append(section_text)

    return chunks

In [9]:
def build_chunks(documents, strategy="simple", **kwargs):
    """Build chunks with one of: simple, paragraph_sliding, section."""
    all_chunks = []

    for doc in documents:
        text = doc["text"]

        if strategy == "simple":
            pieces = simple_fixed_chunking(
                text,
                chunk_size=kwargs.get("chunk_size", 1000),
                overlap=kwargs.get("overlap", 200),
            )
        elif strategy == "paragraph_sliding":
            pieces = paragraph_sliding_chunking(
                text,
                paragraph_window=kwargs.get("paragraph_window", 3),
                paragraph_step=kwargs.get("paragraph_step", 2),
            )
        elif strategy == "section":
            pieces = section_chunking(text)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        for idx, piece in enumerate(pieces):
            all_chunks.append(
                {
                    "chunk_id": f'{doc["id"]}-{strategy}-c{idx}',
                    "repo": doc["repo"],
                    "filename": doc["filename"],
                    "chunk_index": idx,
                    "strategy": strategy,
                    "text": piece,
                }
            )

    return all_chunks

In [10]:
all_docs = []
all_docs += normalize_docs(dtc_langchain, "langchain")
all_docs += normalize_docs(dtc_microsoft, "semantic-kernel")
#all_docs += normalize_docs(dtc_openai, "openai-cookbook")

simple_chunks = build_chunks(all_docs, strategy="simple", chunk_size=1000, overlap=200)
paragraph_chunks = build_chunks(
    all_docs,
    strategy="paragraph_sliding",
    paragraph_window=3,
    paragraph_step=2,
)
section_chunks = build_chunks(all_docs, strategy="section")

print("Docs:", len(all_docs))
print("Simple chunks:", len(simple_chunks))
print("Paragraph+sliding chunks:", len(paragraph_chunks))
print("Section chunks:", len(section_chunks))

all_chunks_by_strategy = {
    "simple": simple_chunks,
    "paragraph_sliding": paragraph_chunks,
    "section": section_chunks,
}

for name, chunks in all_chunks_by_strategy.items():
    avg_len = sum(len(c["text"]) for c in chunks) / max(len(chunks), 1)
    print(f"{name} average chunk length: {avg_len:.1f}")

Docs: 259
Simple chunks: 2184
Paragraph+sliding chunks: 4424
Section chunks: 2383
simple average chunk length: 950.6
paragraph_sliding average chunk length: 550.3
section average chunk length: 704.6


In [11]:
def preview_chunks(chunks, n=3, max_chars=350):
    for i, ch in enumerate(chunks[:n]):
        print(f"[{i}] {ch['repo']} | {ch['filename']} | len={len(ch['text'])}")
        print(ch["text"][:max_chars].replace("\n", " "))
        print("-" * 80)

print("\n=== SIMPLE SAMPLE ===")
preview_chunks(simple_chunks, n=2)

print("\n=== PARAGRAPH + SLIDING SAMPLE ===")
preview_chunks(paragraph_chunks, n=2)

print("\n=== SECTION SAMPLE ===")
preview_chunks(section_chunks, n=2)


=== SIMPLE SAMPLE ===
[0] langchain | langchain-ai-langchain-e207685/.devcontainer/README.md | len=1000
# Dev container  This project includes a [dev container](https://containers.dev/), which lets you use a container as a full-featured dev environment.  You can use the dev container configuration in this folder to build and run the app without needing to install any of its tools locally! You can use it in [GitHub Codespaces](https://github.com/featu
--------------------------------------------------------------------------------
[1] langchain | langchain-ai-langchain-e207685/.devcontainer/README.md | len=1000
ngchain-ai/langchain>. 1. Click on the **Codespaces** tab. 1. Click **Create codespace on master**.  For more info, check out the [GitHub documentation](https://docs.github.com/en/free-pro-team@latest/github/developing-online-with-codespaces/creating-a-codespace#creating-a-codespace).  ## VS Code Dev Containers  [![Open in Dev Containers](https://i
------------------------------

In [12]:
# Search layer: text search, vector search, and hybrid search
import numpy as np
from sentence_transformers import SentenceTransformer
from minsearch import Index, VectorSearch

# Choose one chunking strategy for retrieval.
search_chunks = section_chunks

# 1) Text search (exact/keyword matching)
text_index = Index(
    text_fields=["text", "filename", "repo"],
    keyword_fields=["repo", "strategy"],
)
text_index.fit(search_chunks)

# 2) Vector search (semantic similarity)
embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

chunk_embeddings = []
for d in search_chunks:
    v = embedding_model.encode(d["text"])
    chunk_embeddings.append(v)

chunk_embeddings = np.array(chunk_embeddings)

vector_index = VectorSearch()
vector_index.fit(chunk_embeddings, search_chunks)

# 3) Hybrid search (combine text + vector)
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vector_index.search(q, num_results=num_results)


def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)

    # Weighted Reciprocal Rank Fusion (RRF)
    scores = {}
    items = {}

    for rank, item in enumerate(text_results, start=1):
        key = item["chunk_id"]
        items[key] = item
        scores[key] = scores.get(key, 0.0) + 0.7 * (1.0 / (60 + rank))

    for rank, item in enumerate(vector_results, start=1):
        key = item["chunk_id"]
        items[key] = item
        scores[key] = scores.get(key, 0.0) + 1.0 * (1.0 / (60 + rank))

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [items[key] for key, _ in ranked[:num_results]]


# Quick smoke test
query = "How do I use tools with an agent in LangChain?"
print("TEXT SEARCH")
for r in text_search(query, num_results=3):
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")

print("\nVECTOR SEARCH")
for r in vector_search(query, num_results=3):
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")

print("\nHYBRID SEARCH")
for r in hybrid_search(query, num_results=3):
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

TEXT SEARCH
- langchain | langchain-ai-langchain-e207685/README.md | langchain-4-section-c4
- langchain | langchain-ai-langchain-e207685/README.md | langchain-4-section-c2
- langchain | langchain-ai-langchain-e207685/README.md | langchain-4-section-c5

VECTOR SEARCH
- langchain | langchain-ai-langchain-e207685/README.md | langchain-4-section-c1
- langchain | langchain-ai-langchain-e207685/libs/langchain_v1/README.md | langchain-8-section-c2
- semantic-kernel | microsoft-semantic-kernel-5f282a9/docs/decisions/0070-declarative-agent-schema.md | semantic-kernel-93-section-c21

HYBRID SEARCH
- langchain | langchain-ai-langchain-e207685/README.md | langchain-4-section-c1
- langchain | langchain-ai-langchain-e207685/libs/langchain_v1/README.md | langchain-8-section-c2
- semantic-kernel | microsoft-semantic-kernel-5f282a9/docs/decisions/0070-declarative-agent-schema.md | semantic-kernel-93-section-c21


In [30]:
# Set your key only in local runtime (do NOT commit real keys)


In [15]:
# Day 4: Agent + tools (text/vector/hybrid search)
import json
import os
import httpx
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set for this notebook kernel. "
        "Run `%env OPENAI_API_KEY=...` and re-run this cell."
    )

# For corporate/self-signed TLS environments, point one of these env vars
# to your CA bundle file before running this cell:
#   export SSL_CERT_FILE=/path/to/corp-ca.pem
#   export REQUESTS_CA_BUNDLE=/path/to/corp-ca.pem
ca_bundle = os.getenv("SSL_CERT_FILE") or os.getenv("REQUESTS_CA_BUNDLE")
http_client = httpx.Client(verify=ca_bundle if ca_bundle else True, timeout=60.0)

client = OpenAI(api_key=openai_api_key, http_client=http_client)


def format_context(results, max_chars=1200):
    parts = []
    for i, r in enumerate(results, start=1):
        text = (r.get("text") or "").strip().replace("\n", " ")
        parts.append(
            f"[{i}] repo={r.get('repo')} file={r.get('filename')} chunk_id={r.get('chunk_id')}\n"
            f"{text[:max_chars]}"
        )
    return "\n\n".join(parts)


def search_tool(query, mode="hybrid", num_results=5):
    if mode == "text":
        return text_search(query, num_results=num_results)
    if mode == "vector":
        return vector_search(query, num_results=num_results)
    return hybrid_search(query, num_results=num_results)


def agent_answer(user_query, mode="hybrid", num_results=5, model="gpt-4o-mini"):
    results = search_tool(user_query, mode=mode, num_results=num_results)
    context = format_context(results)

    system_prompt = (
        "You are a helpful technical assistant. "
        "Answer only from provided context. "
        "If context is insufficient, say what is missing."
    )

    prompt = (
        f"User question:\n{user_query}\n\n"
        f"Retrieved context ({mode} search):\n{context}\n\n"
        "Give a concise answer and cite file paths used."
    )

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )

    return {
        "answer": response.output_text,
        "results": results,
    }


# Quick demo
question = "How can I build an agent that uses tools in these repos?"

try:
    out = agent_answer(question, mode="hybrid", num_results=4)
    print(out["answer"])
    print("\nTop sources:")
    for r in out["results"]:
        print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")
except Exception as e:
    print("OpenAI request failed:", e)
    print("If you are on a corporate network, set SSL_CERT_FILE or REQUESTS_CA_BUNDLE to your CA bundle path.")


To build an agent that uses tools in the provided repositories, you can follow these steps:

1. **Define the Agent**: Create or modify the agent configurations in `AgentDefinition.yaml` or `AgentWithRagDefinition.yaml`. Specify the agent's name, instructions, template format, and description.

2. **Configure Tools**: Register the tools your agent will use during its creation. You can do this directly in the agent instantiation call, such as in:
   ```csharp
   AIAgent agent = chatClient.CreateAIAgent(tools: [AIFunctionFactory.Create(GetWeather)]);
   ```
   This allows you to utilize different functions like a weather service in the agent.

3. **Adjust Properties**: Modify properties in the YAML file as per your requirements. These properties include `name`, `template` (for instructions), and `execution_settings` (like temperature).

You can find detailed configuration instructions in:
- `microsoft-semantic-kernel-5f282a9/docs/decisions/0070-declarative-agent-schema.md`
- `microsoft-se

In [16]:
# Day 4 (part 1): OpenAI function calling with search tools
import json

search_tool_schema = {
    "type": "function",
    "name": "search_docs",
    "description": "Search project docs with text, vector, or hybrid mode.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "User query to search in docs",
            },
            "mode": {
                "type": "string",
                "enum": ["text", "vector", "hybrid"],
                "description": "Search mode",
            },
            "num_results": {
                "type": "integer",
                "minimum": 1,
                "maximum": 10,
                "description": "How many snippets to fetch",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}


def run_search_docs(query, mode="hybrid", num_results=5):
    mode = mode or "hybrid"
    num_results = int(num_results or 5)

    if mode == "text":
        results = text_search(query, num_results=num_results)
    elif mode == "vector":
        results = vector_search(query, num_results=num_results)
    else:
        results = hybrid_search(query, num_results=num_results)

    # Keep tool payload compact.
    compact = []
    for r in results:
        compact.append(
            {
                "repo": r.get("repo"),
                "filename": r.get("filename"),
                "chunk_id": r.get("chunk_id"),
                "text": (r.get("text") or "")[:1000],
            }
        )
    return compact


def answer_with_function_calling(user_query, model="gpt-4o-mini"):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a technical assistant. For repo/doc questions, call search_docs first. "
                "Then answer using only returned snippets and cite filenames."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    response = client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool_schema],
    )

    # Handle tool calls (single or multiple).
    for item in response.output:
        if getattr(item, "type", None) == "function_call" and item.name == "search_docs":
            args = json.loads(item.arguments)
            tool_result = run_search_docs(**args)
            messages.append(item)
            messages.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(tool_result),
                }
            )

    final = client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool_schema],
    )
    return final.output_text


# Demo
question = "How do these repos implement tool calling for agents?"
print(answer_with_function_calling(question))


The implementation of tool calling for agents in the repositories primarily involves the following key aspects:

1. **OpenAI Function Calling Support**: The implementation details about supporting function calls can be found in the document titled "OpenAI Function Calling Support" within the semantic-kernel repository (see `microsoft-semantic-kernel-5f282a9/docs/decisions/0017-openai-function-calling.md`).

2. **Function Calling Reliability**: For enhancing the reliability of function calls, refer to the document titled "Function Calling Reliability" in the same repository (see `microsoft-semantic-kernel-5f282a9/docs/decisions/0063-function-calling-reliability.md`).

3. **Legacy Agents**: The document titled "Legacy Agents" mentions the transition from the `Microsoft.SemanticKernel.Experimental.Agents` package to the current Semantic Kernel Agents, which now include support for OpenAI Assistant agents (see `microsoft-semantic-kernel-5f282a9/dotnet/samples/Concepts/Agents/README.md`).



In [17]:
# Day 4 (part 2): PydanticAI agent with search tool
# If needed once: %pip install pydantic-ai
from pydantic_ai import Agent


def pyd_search(query: str, mode: str = "hybrid", num_results: int = 5) -> list[dict]:
    return run_search_docs(query=query, mode=mode, num_results=num_results)


pyd_agent = Agent(
    "openai:gpt-4o-mini",
    system_prompt=(
        "You are a technical assistant for these repositories. "
        "Use the tool to fetch context before answering repo/document questions. "
        "Cite filenames in your response."
    ),
)


@pyd_agent.tool
async def search_docs(ctx, query: str, mode: str = "hybrid", num_results: int = 5) -> list[dict]:
    return pyd_search(query=query, mode=mode, num_results=num_results)


# Demo (Jupyter-safe: use await, not run_sync)
result = await pyd_agent.run("Where can I find examples of tool-based agents in these repos?")
print(result.output)


You can find examples of tool-based agents in the following files within the semantic-kernel repository:

1. **Creating an Agent with access to function tools** - [0070-declarative-agent-schema.md](https://github.com/microsoft/semantic-kernel/blob/main/docs/decisions/0070-declarative-agent-schema.md): This document discusses how to create an agent equipped with function tools and guidance for its behavior.

2. **Getting Started with Agents (Python)** - [getting_started_with_agents/README.md](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/getting_started_with_agents/README.md): This README contains examples relevant to starting with agents.

3. **Getting Started with Agents (DotNet)** - [GettingStartedWithAgents/README.md](https://github.com/microsoft/semantic-kernel/blob/main/dotnet/samples/GettingStartedWithAgents/README.md): This README also includes examples for getting started with agents, tailored for the .NET environment.

4. **Declarative Agent Concept Sam

In [18]:
# Day 5: Retrieval evaluation for text/vector/hybrid
import random
import pandas as pd


def build_project_eval_set(chunks, max_items=120, seed=42):
    rng = random.Random(seed)
    selected = [c for c in chunks if (c.get("text") or "").strip()]
    rng.shuffle(selected)
    selected = selected[:max_items]

    eval_rows = []
    for c in selected:
        text = c["text"].strip().replace("\n", " ")
        query = " ".join(text.split()[:16])
        eval_rows.append(
            {
                "query": query,
                "gold_chunk_id": c["chunk_id"],
                "gold_filename": c["filename"],
            }
        )
    return eval_rows


def retrieval_metrics(eval_rows, search_fn, k=5):
    hits = 0
    mrr_total = 0.0

    for row in eval_rows:
        results = search_fn(row["query"], num_results=k)
        ranked_ids = [r.get("chunk_id") for r in results]

        rank = None
        for i, cid in enumerate(ranked_ids, start=1):
            if cid == row["gold_chunk_id"]:
                rank = i
                break

        if rank is not None:
            hits += 1
            mrr_total += 1.0 / rank

    n = max(len(eval_rows), 1)
    return {f"hit@{k}": hits / n, "mrr": mrr_total / n, "n": len(eval_rows)}


def eval_all_modes(eval_rows, k=5):
    reports = []
    for name, fn in [
        ("text", text_search),
        ("vector", vector_search),
        ("hybrid", hybrid_search),
    ]:
        r = retrieval_metrics(eval_rows, fn, k=k)
        r["mode"] = name
        reports.append(r)

    return pd.DataFrame(reports).sort_values([f"hit@{k}", "mrr"], ascending=False)


eval_rows = build_project_eval_set(search_chunks, max_items=120, seed=42)
eval_df = eval_all_modes(eval_rows, k=5)
eval_df

,hit@5,mrr,n,mode
2,0.950000,0.849444,120,hybrid
1,0.950000,0.836944,120,vector
0,0.866667,0.767500,120,text


In [19]:
# Day 5: Agent answer quality proxy (token overlap vs retrieved top chunk)
import re


def _norm_tokens(text: str):
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return set(t for t in text.split() if len(t) > 2)


def overlap_f1(a: str, b: str) -> float:
    ta, tb = _norm_tokens(a), _norm_tokens(b)
    if not ta or not tb:
        return 0.0
    inter = len(ta & tb)
    p = inter / len(ta)
    r = inter / len(tb)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)


sample_queries = [row["query"] for row in eval_rows[:10]]
rows = []
for q in sample_queries:
    retrieved = hybrid_search(q, num_results=1)
    top_text = retrieved[0]["text"] if retrieved else ""
    answer = answer_with_function_calling(q)
    rows.append(
        {
            "query": q,
            "answer_vs_top_chunk_f1": overlap_f1(answer, top_text),
        }
    )

quality_df = pd.DataFrame(rows)
print("Avg answer vs context token F1:", round(quality_df["answer_vs_top_chunk_f1"].mean(), 4))
quality_df.head()

Avg answer vs context token F1: 0.3778


,query,answer_vs_top_chunk_f1
0,#### Trusted Input Variables ```csharp var cha...,0.205128
1,# Install SK and all dependencies uv sync --al...,0.424242
2,### Google | Priority | Model | Status | Inter...,0.131148
3,# PowerShell $env:Postgres__ConnectionString =...,0.301075
4,## [New model] Option 1.1 - A class per functi...,0.500000


In [20]:
# Day 6: Publish your agent with Streamlit Cloud
# Files added for deployment:
# - project/streamlit_app.py
# - project/requirements.txt

# Local run command:
# !streamlit run project/streamlit_app.py

# Streamlit Cloud settings:
# - Repository: this repo
# - Main file path: project/streamlit_app.py
# - Python: 3.11+ recommended on Streamlit Cloud
# - Secrets: add OPENAI_API_KEY in app settings

print("Day 6 setup complete: use Streamlit Cloud with project/streamlit_app.py")

Day 6 setup complete: use Streamlit Cloud with project/streamlit_app.py


In [21]:
# Day 7: Share results (benchmark and export artifacts)
import json
from datetime import datetime


benchmark_questions = [
    "How do I build an agent with tools?",
    "Where are examples of function calling in these repositories?",
    "How can I add memory or state to an agent workflow?",
    "What are common patterns for retrieval-augmented generation?",
    "How do I evaluate an agent for relevance and grounding?",
]


def run_benchmark(questions):
    rows = []
    for q in questions:
        text_top = text_search(q, num_results=1)
        vector_top = vector_search(q, num_results=1)
        hybrid_top = hybrid_search(q, num_results=1)

        rows.append(
            {
                "question": q,
                "text_top_file": text_top[0]["filename"] if text_top else None,
                "vector_top_file": vector_top[0]["filename"] if vector_top else None,
                "hybrid_top_file": hybrid_top[0]["filename"] if hybrid_top else None,
                "text_vs_vector_same": bool(text_top and vector_top and text_top[0]["chunk_id"] == vector_top[0]["chunk_id"]),
            }
        )
    return pd.DataFrame(rows)


benchmark_df = run_benchmark(benchmark_questions)
benchmark_df

,question,text_top_file,vector_top_file,hybrid_top_file,text_vs_vector_same
0,How do I build an agent with tools?,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/docs/decisio...,False
1,Where are examples of function calling in thes...,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/dotnet/sampl...,microsoft-semantic-kernel-5f282a9/dotnet/sampl...,False
2,How can I add memory or state to an agent work...,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/docs/decisio...,False
3,What are common patterns for retrieval-augment...,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/docs/decisio...,microsoft-semantic-kernel-5f282a9/docs/decisio...,True
4,How do I evaluate an agent for relevance and g...,microsoft-semantic-kernel-5f282a9/python/sampl...,microsoft-semantic-kernel-5f282a9/python/sampl...,microsoft-semantic-kernel-5f282a9/python/sampl...,False


In [29]:
# Day 7: Generate publish-ready summary files
from datetime import datetime, UTC
from pathlib import Path

run_ts = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S")

summary = {
    "run_timestamp_utc": run_ts,
    "retrieval_eval": eval_df.to_dict(orient="records") if "eval_df" in globals() else [],
    "answer_quality_mean_f1": float(quality_df["answer_vs_top_chunk_f1"].mean()) if "quality_df" in globals() else None,
    "benchmark": benchmark_df.to_dict(orient="records"),
    "notes": "Day 7 share package from project notebook",
}

# Save into the current project directory regardless of execution cwd.
out_dir = Path.cwd()
benchmark_path = out_dir / f"day7_benchmark_{run_ts}.csv"
summary_path = out_dir / f"day7_summary_{run_ts}.json"

benchmark_df.to_csv(benchmark_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print("-", benchmark_path)
print("-", summary_path)

post_template = f"""
## Day 7 Results Snapshot

- Retrieval best mode (by hit@5/MRR): check `eval_df`
- Mean answer-vs-context token F1: {summary['answer_quality_mean_f1']}
- Benchmark questions evaluated: {len(benchmark_df)}
- Artifacts: `{benchmark_path}`, `{summary_path}`
""".strip()

print("\n" + post_template)


Saved:
- /Users/NaveenkumarVasudevan/Downloads/Project/aihero/project/day7_benchmark_2026-03-31_02-35-33.csv
- /Users/NaveenkumarVasudevan/Downloads/Project/aihero/project/day7_summary_2026-03-31_02-35-33.json

## Day 7 Results Snapshot

- Retrieval best mode (by hit@5/MRR): check `eval_df`
- Mean answer-vs-context token F1: 0.37783777949223857
- Benchmark questions evaluated: 5
- Artifacts: `/Users/NaveenkumarVasudevan/Downloads/Project/aihero/project/day7_benchmark_2026-03-31_02-35-33.csv`, `/Users/NaveenkumarVasudevan/Downloads/Project/aihero/project/day7_summary_2026-03-31_02-35-33.json`


In [28]:
# Day 7: Final share-ready report (copy/paste)
from pathlib import Path
import json

APP_LINK = "https://nevin7771-ai-hero-projectstreamlit-app-p2uerj.streamlit.app/"
REPO_LINK = "https://github.com/nevin7771/ai-hero"

if "eval_df" not in globals() or "benchmark_df" not in globals():
    raise ValueError("Run Day 5 and Day 7 cells first to populate eval_df and benchmark_df")

best_row = eval_df.iloc[0].to_dict()
best_mode = best_row.get("mode")
best_hit5 = round(float(best_row.get("hit@5", 0.0)), 4)
best_mrr = round(float(best_row.get("mrr", 0.0)), 4)

quality_mean = None
if "quality_df" in globals() and not quality_df.empty:
    quality_mean = round(float(quality_df["answer_vs_top_chunk_f1"].mean()), 4)

share_text = f"""
🚀 Day 7 of building my AI agent: sharing results!

This week I built and shipped a docs assistant over LangChain, Semantic Kernel, and OpenAI Cookbook repositories.

What I measured:
- Best retrieval mode: {best_mode}
- Hit@5: {best_hit5}
- MRR: {best_mrr}
- Answer-vs-context token F1: {quality_mean}
- Benchmark questions: {len(benchmark_df)}

Live app: {APP_LINK}
Repo: {REPO_LINK}

Big takeaway: hybrid retrieval + tool calling gave the most reliable answers with traceable sources.

Following along with AI Hero - who else is building AI agents?
""".strip()

print(share_text)

final_report = {
    "best_mode": best_mode,
    "hit_at_5": best_hit5,
    "mrr": best_mrr,
    "answer_vs_context_token_f1": quality_mean,
    "benchmark_questions": len(benchmark_df),
    "app_link": APP_LINK,
    "repo_link": REPO_LINK,
    "share_text": share_text,
}

out_dir = Path.cwd()
final_path = out_dir / f"day7_final_share_{run_ts}.json"
with open(final_path, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)

print("\nSaved final share artifact:", final_path)

🚀 Day 7 of building my AI agent: sharing results!

This week I built and shipped a docs assistant over LangChain, Semantic Kernel, and OpenAI Cookbook repositories.

What I measured:
- Best retrieval mode: hybrid
- Hit@5: 0.95
- MRR: 0.8494
- Answer-vs-context token F1: 0.3778
- Benchmark questions: 5

Live app: https://nevin7771-ai-hero-projectstreamlit-app-p2uerj.streamlit.app/
Repo: https://github.com/nevin7771/ai-hero

Big takeaway: hybrid retrieval + tool calling gave the most reliable answers with traceable sources.

Following along with AI Hero - who else is building AI agents?

Saved final share artifact: /Users/NaveenkumarVasudevan/Downloads/Project/aihero/project/day7_final_share_2026-03-31_02-31-16.json
